# Mejoras para IES_Teis_AI_Triage


El reto oficial pide dos mejoras:

1. Procesar nuevos correos sin depender de un JSON fijo.
2. Mejorar la recuperación tipo RAG 2.0.

La primera mejora es mucho más viable en examen offline porque no necesita modelos nuevos.


In [ ]:
# Mejora offline: procesar todos los JSON de una carpeta inbox/.
# Luego moverlos a processed/ o failed/ para no reprocesar.

from pathlib import Path
import json
import shutil

INBOX_DIR = Path("inbox")
PROCESSED_DIR = Path("processed")
FAILED_DIR = Path("failed")

for directory in [INBOX_DIR, PROCESSED_DIR, FAILED_DIR]:
    directory.mkdir(exist_ok=True)


def cargar_correos_pendientes():
    correos = []
    for json_path in sorted(INBOX_DIR.glob("*.json")):
        try:
            data = json.loads(json_path.read_text(encoding="utf-8"))
            if isinstance(data, dict):
                data = [data]
            for correo in data:
                correo["source_file"] = json_path.name
                correos.append(correo)
            shutil.move(str(json_path), PROCESSED_DIR / json_path.name)
        except Exception:
            shutil.move(str(json_path), FAILED_DIR / json_path.name)
    return correos

correos = cargar_correos_pendientes()
print("Correos cargados:", len(correos))


In [ ]:
# Mejora: evitar duplicados por ID de correo.
# Si un correo ya fue procesado, se salta.

import csv
from pathlib import Path

PROCESSED_IDS = Path("processed_ids.txt")


def leer_ids_procesados():
    if not PROCESSED_IDS.exists():
        return set()
    return set(PROCESSED_IDS.read_text(encoding="utf-8").splitlines())


def guardar_id_procesado(correo_id):
    with PROCESSED_IDS.open("a", encoding="utf-8") as file:
        file.write(str(correo_id) + "\n")

ids_procesados = leer_ids_procesados()
correos_nuevos = [correo for correo in correos if str(correo.get("id")) not in ids_procesados]
print("Correos nuevos:", len(correos_nuevos))


RAG 2.0 viable si ya está todo descargado:

- Multi-query: generar varias versiones de la pregunta antes de buscar.
- Reranking: recuperar más documentos y reordenarlos con un cross-encoder.
- Routing: si el correo es saludo o agradecimiento, responder sin RAG.

RAG 2.0 no viable si no están cacheados:

- descargar embeddings nuevos
- descargar reranker de Hugging Face
- usar Ollama si no hay modelo local
